In [ ]:
%load_ext autoreload
%autoreload 2

import sys

sys.path.append("../")
from odometry.estimators.estimators import (
    KalmanXYPhiSpeedKinematic,
    KalmanXYPhiSpeedGyroEncoder,
    InertialIntegrator,
)

import numpy as np
from scipy.interpolate import splrep, BSpline
import matplotlib.pyplot as plt

## Generate trajectories

In [ ]:
def wrap_heading(heading_rad):
    """wraps the heading (phi) to be between [-pi,pi]

    Args:
        heading_rad (_type_): the heading in radians

    Returns:
        _type_: the wrapped heading in radians
    """
    # implement wrapping around when abs(heading) > pi
    if np.abs(heading_rad) > np.pi:
        return -1 * np.sign(heading_rad) * (2 * np.pi - np.abs(heading_rad))

    else:
        return heading_rad


def gen_random_traj(
    dt=0.1, t_max=100, markov_speed=1e-2, markov_heading=5e-2, bias_heading=1e-3
):
    # state is x, y, phi, speed
    R_start = np.diag([100, 100, 1, 5])
    state = R_start @ np.random.randn(4)
    state[3] = abs(state[3])

    # at each step, propagate, add markov to heading and speed
    n_steps = int(t_max // dt)
    t_history = np.linspace(0, t_max, n_steps)

    # do some smoothing on heading and speed
    heading_history = [state[2]]
    speed_history = [state[3]]
    for _ in t_history[1:]:
        speed_history.append(
            max(0, speed_history[-1] + dt * markov_speed * np.random.randn())
        )
        heading_history.append(
            wrap_heading(
                heading_history[-1]
                + dt * markov_heading * np.random.randn()
                + bias_heading * dt
            )
        )
    tck_s = splrep(t_history, np.asarray(speed_history), s=1)
    tck_h = splrep(t_history, np.asarray(heading_history), s=1)
    sp_s = BSpline(*tck_s)
    sp_h = BSpline(*tck_h)

    # compute the positions
    state_history = [state.copy()]
    new_state = state.copy()
    for i in range(1, len(t_history)):
        # get average phi and speed between times
        avg_phi = (sp_h(t_history[i]) + sp_h(t_history[i - 1])) / 2  # approximation...
        avg_speed = (
            sp_s(speed_history[i]) + sp_s(speed_history[i - 1])
        ) / 2  # approximation...

        # Update states
        new_state[0] = new_state[0] + dt * avg_speed * np.cos(avg_phi)
        new_state[1] = new_state[1] + dt * avg_speed * np.sin(avg_phi)
        new_state[2] = sp_h(t_history[i])
        new_state[3] = (
            np.sqrt(
                (new_state[0] - state_history[-1][0]) ** 2
                + (new_state[1] - state_history[-1][1]) ** 2
            )
            / dt
        )
        state_history.append(new_state.copy())

    return np.asarray(t_history), np.asarray(state_history)

In [ ]:
def plot_result(t_history, x_history, axs, p_history=None, add_dots=False):
    if add_dots:
        axs[0].plot(x_history[0, 0], x_history[0, 1], "ro", label="start")
        axs[0].plot(x_history[-1, 0], x_history[-1, 1], "go", label="end")
        axs[0].legend()
    axs[0].plot(x_history[:, 0], x_history[:, 1])
    # states
    for i, title in zip(range(1, 5), ["X", "Y", "Phi", "Speed"]):
        axs[i].plot(t_history, x_history[:, i - 1])
        if p_history is not None:
            axs[i].plot(t_history, x_history[:, i - 1] + p_history[:, i - 1], "r--")
            axs[i].plot(t_history, x_history[:, i - 1] - p_history[:, i - 1], "r--")
        axs[i].set_title(title)
    # biases, if available
    if len(x_history[0, :]) > 4:
        for i, title in zip(range(5, 7), ["Gyro Bias", "Encoder Bias"]):
            axs[i].plot(t_history, x_history[:, i - 1])
            if p_history is not None:
                axs[i].plot(t_history, x_history[:, i - 1] + p_history[:, i - 1], "r--")
                axs[i].plot(t_history, x_history[:, i - 1] - p_history[:, i - 1], "r--")
            axs[i].set_title(title)

In [ ]:
# generate a trajectory
np.random.seed(1)
dt = 0.01
t_max = 500
t_history, state_history = gen_random_traj(
    dt=dt, t_max=t_max, markov_heading=1e-1, markov_speed=5e-1, bias_heading=1e-2
)

# visualize trajectory
fig, axs = plt.subplots(1, 5, figsize=(16, 5))
plot_result(t_history, state_history, axs, add_dots=False)

plt.show()

## Run a simple Kalman filter experiment with a random trajectory

In [ ]:
# measurement function
from typing import List


def imu_func(x: np.ndarray, x_last: np.ndarray, dt: float, g_bias=0):
    """Generates imu measurements with a super simple model"""
    gyro = g_bias + (x[2] - x_last[2]) / dt  # heading difference
    accel = None
    return gyro, accel


def encoder_func(x: np.ndarray, x_last: np.ndarray, dt: float, e_bias=0):
    sencode = e_bias + (x[3] + x_last[3]) / 2  # average velocity
    return sencode


def h_func(x: np.ndarray, msmt_components: List[str]):
    z = []
    for component in msmt_components:
        if component == "x":
            z.append(x[0])
        elif component == "y":
            z.append(x[1])
        elif component == "phi":
            z.append(x[2])
        elif component == "speed":
            z.append(x[3])
        elif component == "vx":
            raise NotImplementedError
        elif component == "vy":
            raise NotImplementedError
        else:
            raise NotImplementedError(component)
    if len(z) == 0:
        raise RuntimeError(f"Did not populate z using {msmt_components}")
    z = np.asarray(z)
    return z

In [ ]:
# initialization
filter_type = "inertial"  # "inertial"
t0 = t_history[0]
inertial_integrator = InertialIntegrator()
if filter_type == "kinematic":
    # state vector is [x, y, phi, speed]
    x0 = state_history[0, :].copy()
    P0 = np.diag([5, 5, 0.1, 1])
    navigator = KalmanXYPhiSpeedKinematic(t0, x0, P0, do_chi2=True)
elif filter_type == "inertial":
    # state vector is [x, y, phi, speed, gyro bias, encoder bias]
    x0 = np.array([*(state_history[0, :].copy()), 0, 0])
    P0 = np.diag([5, 5, 0.1, 1, 1e-2, 1e-2])
    navigator = KalmanXYPhiSpeedGyroEncoder(t0, x0, P0, do_chi2=True)
else:
    raise NotImplementedError(filter_type)

# msmt_components = ["x", "y"]
# R = np.diag([1, 1]) ** 2
msmt_components = ["x", "y", "phi"]
R = np.diag([1, 1, 0.1]) ** 2
# msmt_components = ["x", "y", "phi", "speed"]
# R = np.diag([1, 1, 0.1, 1]) ** 2

# rates
EPS = 1e-6
t_last_imu = 0
i_last_imu = 0
rate_imu = 100
i_last_encoder = 0
t_last_encoder = 0
rate_encoder = 10
t_last_msmt = 0
rate_msmt = 5

# this is for generating measurements for the trajectory
cR = np.linalg.cholesky(R)

# run experiment over time
est_history = [x0.copy()]
g_history = [0]
y_history = [np.array([0, 0])]
p_history = [np.sqrt(np.diag(P0))]
n_time = 50000
for i, (t, state) in enumerate(zip(t_history[:n_time], state_history[:n_time])):
    if i == 0:
        continue

    # inertials
    if (t - t_last_imu + EPS) > 1 / rate_imu:
        dt_imu = t - t_last_imu
        gyro, accel = imu_func(state, state_history[i_last_imu], dt_imu)
        inertial_integrator.add_gyro(dt_imu, gyro)
        inertial_integrator.add_accel(dt_imu, accel)
        i_last_imu = i
        t_last_imu = t

    # encoder
    if (t - t_last_encoder + EPS) > 1 / rate_encoder:
        dt_encoder = t - t_last_encoder
        encoder = encoder_func(state, state_history[i_last_encoder], dt_encoder)
        i_last_encoder = i
        t_last_encoder = t

        # integrate the imu data
        dt_imu, inertial = inertial_integrator.integrate()
        assert np.isclose(dt_imu, dt_encoder)
        inertial.sencode = encoder

        # apply the prediction on every encoder measurement
        navigator.predict(dt_imu, inertial, check_P=False)

    # updates
    if (t - t_last_msmt + EPS) > 1 / rate_msmt:
        # measurement model
        z = h_func(state, msmt_components) + cR @ np.random.randn(len(msmt_components))

        # run filter
        navigator.update(t, z, R, msmt_components)
        t_last_msmt = t

        # store filter states
        g_history.append(navigator.g)
        y_history.append(navigator.y)

    # store filter states
    est_history.append(navigator.x.copy())
    p_history.append(np.sqrt(np.diag(navigator.P)))


est_history = np.asarray(est_history)
p_history = np.asarray(p_history)

# visualize trajectory
if navigator.n_states == 4:
    fig, axs = plt.subplots(1, 5, figsize=(16, 5))
else:
    fig, axs = plt.subplots(1, 7, figsize=(16, 5))

plot_result(
    t_history[:i],
    state_history[:i, :],
    axs,
    p_history=p_history[:i, :],
    add_dots=False,
)
plot_result(
    t_history[:i],
    est_history[:i, :],
    axs,
    p_history=p_history[:i, :],
    add_dots=False,
)
plt.show()

# visualize chi2 result
plt.plot(g_history[:i], label="test statistic")
plt.axhline(
    navigator.g_thresh[len(z)], color="green", linestyle="--", label="chi2 threshold"
)
plt.legend()
plt.show()

plt.hist([y[0] for y in y_history], bins=40, alpha=0.5)
plt.hist([y[1] for y in y_history], bins=40, alpha=0.5)
plt.show()